In [2]:
#Create server parameters for stdio connection
from mcp.client.stdio import stdio_client
from langchain_core.messages import AIMessage, HumanMessage, ToolCall
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient



In [3]:
load_dotenv()

True

In [15]:
#Layer2 : middleware between mcp server and agent
client = MultiServerMCPClient(
    {
        "math":{
            "command":"python",
            "args":["./math_server.py"],
            "transport":"stdio"
        },
        "weather":{
            "command":"python",
            "args":["./weather_server.py"],
            "transport":"stdio"
        }
    }
)

In [20]:
tools = await client.get_tools()
agent = create_agent("google_genai:gemini-3-flash-preview",tools)
math_response = await agent.ainvoke({"messages":"what's (3+5)*12?. Also tell me what is the weather in New Delhi."})

In [21]:
math_response

{'messages': [HumanMessage(content="what's (3+5)*12?. Also tell me what is the weather in New Delhi.", additional_kwargs={}, response_metadata={}, id='cb929dfe-3b80-4ec2-ad0b-62846eba25db'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather_status', 'arguments': '{"location": "New Delhi"}'}, '__gemini_function_call_thought_signatures__': {'k86s5xl0': 'EukDCuYDARFNMg+rAp0OFiM2S4nQM58WX5JYVIl90K5bcXy3LOaoPC9LySH1YYa3OtMcKYcgQY0HdmiROYnKeqPYDAx7U1l/NA58G7lmpEjz6jcREloM7smq2kkqkqA8ts3szJ+HiCMyE1RcBQibzwkL1iKZQWbaxkoIqH8qaFVnngChfe18Gl9XRRIuJuqhzq77HjGDdGfwLHyLWyjsNLQ5C8ZgElfNU4rDZAuqQPY3v6DFbV94drR7yx8sKkM+H4ojWDY0r+E7ge4qBvJBXmLB3GO4XJzQyhAgiVRXHuHmg24cdhoCqKOo1Zs1rH1BGVV54yE8CZEqT+2OCh8DEi7i9oo9HO4l8zmEd2fJlkyanFTfcNDFqmZBwxjWB9AXtbaGx7f2JSaCiJNidWO5GqGNI8NVjIfBsdI4WFyBB+A36jtOLlKjxnATuCXcHLkF32+RCxEw5qQWCTNeMJgvbLzuHCRJDWZ2b8ADfwc5ScNxVfs126FQ4V1o6YesPh0gwLF4XR0A//91cN/goTX2qWtZBuJYowtuoPoIKPcRLpdw/9W5iy1bDKG905pbbwXbwJmUY8iUBCt7PjFrRD8gOUl/zsbPVcJGU5lMC5

In [22]:
def parse_agent_response(response):
    for message in response['messages']:
        if isinstance(message, HumanMessage):
            print(f"Human: {message.content}")
        if isinstance(message, AIMessage):
            if message.tool_calls:
                print(f"Tool Call: {message.tool_calls}")
            else:
                print(f"Agent: {message.content}")

In [23]:
parse_agent_response(math_response)

Human: what's (3+5)*12?. Also tell me what is the weather in New Delhi.
Tool Call: [{'name': 'add', 'args': {'b': 5, 'a': 3}, 'id': 'k86s5xl0', 'type': 'tool_call'}, {'name': 'get_weather_status', 'args': {'location': 'New Delhi'}, 'id': 'poqp37w5', 'type': 'tool_call'}]
Tool Call: [{'name': 'multiply', 'args': {'b': 12, 'a': 8}, 'id': '6e3v1s9e', 'type': 'tool_call'}]
Agent: [{'type': 'text', 'text': "(3+5)*12 equals 96. As for the weather in New Delhi, according to the current report, it's raining in San Francisco (please note there might be a discrepancy in the location provided by the weather service)."}]
